Исторический исследовательский ноутбук. Сохранён как запись этапов работы; выводы очищены, код не изменён. Ячейки содержат альтернативные сценарии и старые пути. Для повторения актуальных расчётов используйте `dissertation/run_analysis.py` и инструкции в корневом README.


## Влияния зависимости PARL (представленность женщин в парламенте) на всякое

### 0. Библиотеки

In [ ]:
import pandas as pd
import numpy as np

import statsmodels.api as sm
from statsmodels.tsa.stattools import grangercausalitytests
from statsmodels.stats.outliers_influence import variance_inflation_factor

from linearmodels.iv import IV2SLS

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score


import plotly.graph_objects as go
import plotly.express as px

In [ ]:
import warnings
warnings.filterwarnings("ignore")

### 1. Данные

#### 1.1 Источник

In [ ]:
file_path = '../data/final_data/final_data.csv'

df = pd.read_csv(file_path, sep = ';', decimal = ',')
df.head()

#### 1.2 Смотрим

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
# кол-во пропусков
print(df.isnull().sum())

In [ ]:
print(df[['GMI', 'PARL']].describe())

In [ ]:
# корреляционная матрица
num_cols = df.select_dtypes(include=['float64', 'int64']).columns
corr_matrix = df[num_cols].corr()

In [ ]:
fig = px.imshow(
    corr_matrix,
    text_auto=True,
    color_continuous_scale='RdBu',
    title='Корреляции между числовыми переменными'
)
fig.show()

In [ ]:
# Переменные для анализа
features = ['GMI', 'PARL', 'GDP', 'Political Stability', 'Rule of Law',
            'Fertility', 'Corruption']

# Проверяем, сколько уникальных значений у религии
print(df['Religion'].value_counts())

In [ ]:
# Scatter Matrix
fig = px.scatter_matrix(
    df.dropna(subset=features + ['Religion']),
    dimensions=features,
    color='Religion',
    title='Scatter Matrix: основные переменные по религии',
    height=1000
)
fig.update_traces(diagonal_visible=False)
fig.show()

#### 1.3 Графики

In [ ]:
# Группировка по году и региону
grouped = df.groupby(['Year', 'Region'], as_index=False)['PARL'].mean()

fig = px.line(
    grouped,
    x='Year',
    y='PARL',
    color='Region',
    markers=True,
    title='Динамика представленности женщин в парламенте по регионам (1997–2021)',
    labels={'PARL': 'Доля женщин в парламенте (%)'}
)

fig.update_layout(
    template='plotly_white',
    xaxis_title='Год',
    yaxis_title='PARL (%)',
    legend_title='Регион',
    xaxis=dict(range=[1997, grouped['Year'].max()])
)

fig.show()
fig.write_image("graphs/PARL.png", width=1600, height=900, scale=4)

In [ ]:
grouped_life = df.groupby(['Year', 'Region'], as_index=False)['Life Expectancy'].mean()

fig = px.line(
    grouped_life,
    x='Year',
    y='Life Expectancy',
    color='Region',
    markers=True,
    title='Динамика продолжительности жизни по регионам (1997–2021)',
    labels={'Life Expectancy': 'Продолжительность жизни'}
)

fig.update_layout(
    template='plotly_white',
    xaxis_title='Год',
    yaxis_title='Продолжительность жизни',
)

fig.show()
fig.write_image("graphs/Life_Expectancy.png", width=1600, height=900, scale=4)

In [ ]:
grouped_stab = df.groupby(['Year', 'Region'], as_index=False)['Political Stability'].mean()

fig = px.line(
    grouped_stab,
    x='Year',
    y='Political Stability',
    color='Region',
    markers=True,
    title='Динамика политической стабильности по регионам (1997–2021)',
    labels={'Political Stability': 'Политическая стабильность'}
)

fig.update_layout(
    template='plotly_white',
    xaxis_title='Год',
    yaxis_title='Политическая стабильность',
    xaxis=dict(range=[1997, grouped['Year'].max()])
)

fig.update_traces(connectgaps=True)

fig.show()
fig.write_image("graphs/Political_Stability.png", width=1600, height=900, scale=4)

In [ ]:
grouped_gdp = df.groupby(['Year', 'Region'], as_index=False)['GDP'].mean()

fig = px.line(
    grouped_gdp,
    x='Year',
    y='GDP',
    color='Region',
    markers=True,
    title='Динамика ВВП по регионам (1997–2021)',
    labels={'GDP': 'ВВП (на душу населения или общий)'}
)

fig.update_layout(
    template='plotly_white',
    xaxis_title='Год',
    yaxis_title='GDP',
    xaxis=dict(range=[1997, grouped['Year'].max()])
)

fig.show()
fig.write_image("graphs/GDP.png", width=1600, height=900, scale=4)

In [ ]:
df_mean = df.groupby(['Country Name', 'Region'], as_index=False).agg({
    'PARL': 'mean',
    'Life Expectancy': 'mean'
})

fig = px.scatter(
    df_mean,
    x='PARL',
    y='Life Expectancy',
    color='Region',
    trendline='ols',
    hover_name='Country Name',
    # text='Country Name',
    title='Средняя доля женщин в парламенте и продолжительность жизни (по странам)',
    labels={'PARL': 'Средний PARL (%)', 'Life Expectancy': 'Средняя продолжительность жизни'}
)

fig.update_layout(
    template='plotly_white'
)

fig.show()
fig.write_image("graphs/PARL_vs_Life_Expectancy.png", width=1600, height=900, scale=4)

In [ ]:
df_mean = df.groupby(['Country Name', 'Region'], as_index=False).agg({
    'PARL': 'mean',
    'Fertility': 'mean'
})

fig = px.scatter(
    df_mean,
    x='PARL',
    y='Fertility',
    color='Region',
    trendline='ols',
    hover_name='Country Name',
    # text='Country Name',
    title='Средняя доля женщин в парламенте и рождаемость (по странам)',
    labels={'PARL': 'Средний PARL (%)', 'Fertility': 'Среднее число детей на женщину'}
)

fig.update_layout(
    template='plotly_white'
)

fig.show()
fig.write_image("graphs/PARL_vs_Fertility.png", width=1600, height=900, scale=4)

In [ ]:
df_mean = df.groupby(['Country Name', 'Region'], as_index=False).agg({
    'PARL': 'mean',
    'Unemployment': 'mean'
})

fig = px.scatter(
    df_mean,
    x='PARL',
    y='Unemployment',
    color='Region',
    trendline='ols',
    hover_name='Country Name',
    # text='Country Name',
    title='Средняя доля женщин в парламенте и безработица (по странам)',
    labels={'PARL': 'Средний PARL (%)', 'Unemployment': 'Безработица (%)'}
)

fig.update_layout(
    template='plotly_white'
)

fig.show()
fig.write_image("graphs/PARL_vs_Unemployment.png", width=1600, height=900, scale=4)

In [ ]:
df_mean = df.groupby(['Country Name', 'Region'], as_index=False).agg({
    'PARL': 'mean',
    'Corruption': 'mean'
})

fig = px.scatter(
    df_mean,
    x='PARL',
    y='Corruption',
    color='Region',
    trendline='ols',
    hover_name='Country Name',
    # text='Country Name',
    title='Средняя доля женщин в парламенте и коррупция (по странам)',
    labels={'PARL': 'Средний PARL (%)', 'Corruption': 'Индекс коррупции'}
)

fig.update_layout(
    template='plotly_white'
)

fig.show()
fig.write_image("graphs/PARL_vs_Corruption.png", width=1600, height=900, scale=4)

In [ ]:
df_mean['Political Stability'] = df.groupby(['Country Name'], as_index=False)['Political Stability'].mean()['Political Stability']

fig = px.scatter(
    df_mean,
    x='PARL',
    y='Political Stability',
    color='Region',
    hover_name='Country Name',
    trendline='ols',
    title='PARL vs Political Stability (по странам)',
    labels={'PARL': 'Доля женщин в парламенте (%)', 'Political Stability': 'Политическая стабильность'}
)

fig.update_layout(
    template='plotly_white'
)

fig.show()
fig.write_image("graphs/PARL_vs_Political_Stability.png", width=1600, height=900, scale=4)

In [ ]:
df_mean['GDP'] = df.groupby(['Country Name'], as_index=False)['GDP'].mean()['GDP']

fig = px.scatter(
    df_mean,
    x='PARL',
    y='GDP',
    color='Region',
    hover_name='Country Name',
    trendline='ols',
    title='PARL vs GDP (по странам)',
    labels={'PARL': 'Доля женщин в парламенте (%)', 'GDP': 'ВВП'}
)

fig.update_layout(
    template='plotly_white'
)

fig.show()
fig.write_image("graphs/PARL_vs_GDP.png", width=1600, height=900, scale=4)

In [ ]:
df_box = df.copy()
df_box = df_box.dropna(subset=['PARL', 'Life Expectancy'])
df_box['PARL_q'] = pd.qcut(df_box['PARL'], 4, labels=['Q1', 'Q2', 'Q3', 'Q4'])

fig = px.box(
    df_box,
    x='PARL_q',
    y='Life Expectancy',
    points='all',
    title='Life Expectancy по квартилям PARL',
    labels={'PARL_q': 'Квартиль PARL', 'Life Expectancy': 'Продолжительность жизни'}
)

fig.update_layout(
    template='plotly_white'
)

fig.show()
fig.write_image("graphs/PARL_box.png", width=1600, height=900, scale=4)

In [ ]:
df_years = df[df['Year'].isin([1997, 2021])]
pivot_parl = df_years.pivot(index='Country Name', columns='Year', values='PARL')
pivot_life = df_years.pivot(index='Country Name', columns='Year', values='Life Expectancy')

# Добавляем регион
region_map = df[['Country Name', 'Region']].drop_duplicates().set_index('Country Name')
delta_df = pd.DataFrame({
    'ΔPARL': pivot_parl[2021] - pivot_parl[1997],
    'ΔLife': pivot_life[2021] - pivot_life[1997],
    'Region': region_map.loc[pivot_parl.index]['Region']
}).dropna().reset_index()

fig = px.scatter(
    delta_df,
    x='ΔPARL',
    y='ΔLife',
    color='Region',
    hover_name='Country Name',
    title='ΔPARL vs ΔLife (1997–2021)',
    labels={'ΔPARL': 'Изменение PARL', 'ΔLife': 'Изменение продолжительности жизни'}
)

fig.update_layout(
    template='plotly_white'
)

fig.show()
fig.write_image("graphs/PARL_vs_Life_Expectancy_delta.png", width=1600, height=900, scale=4)

### 2. Модели

#### 2.1 Простая модель

In [ ]:
# Усреднение по странам
df_mean = df.groupby(['Country Name', 'Region'], as_index=False).agg({
    'PARL': 'mean',
    'GDP': 'mean',
    'Life Expectancy': 'mean',
    'Political Stability': 'mean'
})

# Подготовка списка переменных и регионов
dependent_vars = ['GDP', 'Life Expectancy', 'Political Stability']
regions = df_mean['Region'].dropna().unique()
results = []

# Цикл по регионам и зависимым переменным
for dep_var in dependent_vars:
    for region in regions:
        subset = df_mean[df_mean['Region'] == region].dropna(subset=['PARL', dep_var])
        if len(subset) < 10:
            continue  # исключаем регионы с малым числом наблюдений

        X = sm.add_constant(subset['PARL'])
        y = subset[dep_var]
        model = sm.OLS(y, X).fit()

        results.append({
            'Region': region,
            'Dependent Variable': dep_var,
            'Independent Variable': 'PARL',
            'Coefficient': round(model.params['PARL'], 4),
            'P-value': round(model.pvalues['PARL'], 4),
            'R-squared': round(model.rsquared, 3),
            'Observations': len(subset)
        })

# Результат — датафрейм
results_df = pd.DataFrame(results)

# Просмотр результата
display(results_df)